# Практична робота №3 — Методи нульового порядку багатовимірної безумовної оптимізації

Вкажи номер варіанта, коли клітинка попросить — решта коду підхопить твою функцію, початкову точку, точність та крок автоматично.

In [ ]:
import numpy as np
import sympy as sp
import pandas as pd
import time
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.optimize import minimize

In [ ]:
from variants import VARIANTS

_default = globals().get("VARIANT", 1)
_raw = input(f"Номер варіанта (1-{len(VARIANTS)}) [{_default}]: ").strip()
VARIANT = int(_raw) if _raw else _default
assert VARIANT in VARIANTS, f"Немає варіанта {VARIANT}"

## 1. Побудова моделі

З таблиці варіантів дістаємо функцію (як рядок для SymPy із символами `x1`, `x2`), початкову точку, точність і початковий крок h. `sp.lambdify` перетворює символьний вираз на швидку numpy-функцію, а `f_vec` — тонка обгортка, що приймає точку як масив `[x1, x2]` (зручно для методів прямого пошуку) і рахує кількість викликів.

In [ ]:
expr_str, x0, eps0, h0 = VARIANTS[VARIANT]

x1, x2 = sp.symbols('x1 x2', real=True)
f_sym = sp.sympify(expr_str, locals={'x1': x1, 'x2': x2})
f_np = sp.lambdify((x1, x2), f_sym, 'numpy')

calls = {"n": 0}


def f_vec(p):
    calls["n"] += 1
    return float(f_np(p[0], p[1]))


x0 = np.array(x0, dtype=float)
print(f"Варіант {VARIANT}")
print(f"f(x1, x2) = {f_sym}")
print(f"x0 = {x0}, eps = {eps0}, h = {h0}")

## 2. Лінії рівня — оцінка наявності яру

Будуємо контурний графік навколо початкової точки й очікуваного мінімуму, щоб візуально оцінити форму ландшафту: чи витягнуті лінії рівня в яр під кутом до координатних осей.

In [ ]:
def make_grid(center, half_width=3.0, n=250):
    xs = np.linspace(center[0] - half_width, center[0] + half_width, n)
    ys = np.linspace(center[1] - half_width, center[1] + half_width, n)
    X, Y = np.meshgrid(xs, ys)
    Z = f_np(X, Y)
    return X, Y, Z


X, Y, Z = make_grid(x0, half_width=max(3.0, np.linalg.norm(x0 - 1) + 1.5))
fig0, ax0 = plt.subplots(figsize=(6, 5))
cs = ax0.contour(X, Y, Z, levels=30, cmap='viridis')
ax0.plot(*x0, 'ko', ms=8, label='x0')
ax0.set_xlabel('x1'); ax0.set_ylabel('x2'); ax0.legend()
ax0.set_title(f'Варіант {VARIANT}: лінії рівня f(x1, x2)')
fig0.tight_layout()
fig0.savefig('contour_base.png', dpi=150)
fig0

## 3. Аналітичний розв'язок

Варіант 1 належить до родини узагальненої функції Розенброка `a(x2 - x1^2)^2 + (1 - x1)^2`. Другий доданок дорівнює нулю лише при `x1 = 1`, а тоді перший доданок обнуляється при `x2 = x1^2 = 1` — отже мінімум єдиний і не залежить від коефіцієнта a: `x* = (1, 1)`, `f(x*) = 0`. Дно яру лежить уздовж параболи `x2 = x1^2`, тому жодна з координатних осей не збігається з напрямком спуску вздовж яру.

In [ ]:
x_exact = np.array([1.0, 1.0])
print(f"x* = {x_exact}, f(x*) = {f_np(*x_exact):.10f}")

## 4. Реалізація методів

Кожна функція повертає `(x*, кількість_ітерацій, історія_точок)`. Одновимірна мінімізація в покоординатному спуску — метод золотого перерізу з практичної роботи №2.

In [ ]:
def golden_line(f, x, d, a=-2.0, b=2.0, eps=1e-6):
    """Мінімізація f(x + t*d) за напрямком d методом золотого перерізу."""
    phi = (np.sqrt(5) - 1) / 2
    t1, t2 = b - phi * (b - a), a + phi * (b - a)
    f1, f2 = f(x + t1 * d), f(x + t2 * d)
    while b - a > eps:
        if f1 <= f2:
            b, t2, f2 = t2, t1, f1
            t1 = b - phi * (b - a); f1 = f(x + t1 * d)
        else:
            a, t1, f1 = t1, t2, f2
            t2 = a + phi * (b - a); f2 = f(x + t2 * d)
    return (a + b) / 2


def coordinate_descent(f, x0, eps=1e-6, itmax=5000):
    x = np.array(x0, float); n = len(x); path = [x.copy()]
    for k in range(itmax):
        x_old = x.copy()
        for i in range(n):
            d = np.zeros(n); d[i] = 1.0
            x = x + golden_line(f, x, d) * d
        path.append(x.copy())
        if np.linalg.norm(x - x_old) < eps:
            break
    return x, k + 1, np.array(path)


def hooke_jeeves(f, x0, h=0.5, alpha=2.0, eps=1e-6, itmax=5000):
    def explore(base, h):
        x = base.copy(); fx = f(x)
        for i in range(len(x)):
            for s in (h, -h):
                trial = x.copy(); trial[i] += s
                ft = f(trial)
                if ft < fx:
                    x, fx = trial, ft
                    break
        return x, fx

    x = np.array(x0, float); path = [x.copy()]; k = 0
    while h > eps and k < itmax:
        k += 1
        xn, fn = explore(x, h)
        if fn < f(x):
            while True:  # пошук за зразком
                xp = xn + alpha * (xn - x)
                xe, fe = explore(xp, h)
                if fe < fn:
                    x, xn, fn = xn, xe, fe
                else:
                    x = xn
                    break
            path.append(x.copy())
        else:
            h /= 2.0  # дроблення кроку
    return x, k, np.array(path)

## 5. Запуск трьох методів

In [ ]:
calls["n"] = 0
t0 = time.perf_counter()
x_cd, it_cd, path_cd = coordinate_descent(f_vec, x0, eps=eps0)
t_cd = time.perf_counter() - t0
calls_cd = calls["n"]

calls["n"] = 0
t0 = time.perf_counter()
x_hj, it_hj, path_hj = hooke_jeeves(f_vec, x0, h=h0, eps=eps0)
t_hj = time.perf_counter() - t0
calls_hj = calls["n"]

calls["n"] = 0
t0 = time.perf_counter()
path_nm = [np.array(x0)]
res_nm = minimize(
    f_vec, x0, method='Nelder-Mead',
    options={'xatol': eps0, 'fatol': eps0},
    callback=lambda xk: path_nm.append(np.array(xk)),
)
t_nm = time.perf_counter() - t0
calls_nm = calls["n"]
path_nm = np.array(path_nm)

print(f"{'Метод':<22}{'x1*':>10}{'x2*':>10}{'f(x*)':>14}{'ітер.':>8}{'обчисл. f':>11}{'час, с':>9}")
for name, x_, it_, calls_, t_ in [
    ("Покоординатний спуск", x_cd, it_cd, calls_cd, t_cd),
    ("Хука-Дживса", x_hj, it_hj, calls_hj, t_hj),
    ("Нелдера-Міда", res_nm.x, res_nm.nit, calls_nm, t_nm),
]:
    print(f"{name:<22}{x_[0]:>10.5f}{x_[1]:>10.5f}{f_vec(x_):>14.3e}{it_:>8}{calls_:>11}{t_:>9.4f}")

## 6. Траєкторії пошуку на лініях рівня (три графіки)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (name, path) in zip(axes, [
    ("Покоординатний спуск", path_cd),
    ("Хука-Дживса", path_hj),
    ("Нелдера-Міда", path_nm),
]):
    ax.contour(X, Y, Z, levels=30, cmap='viridis', alpha=.6)
    ax.plot(path[:, 0], path[:, 1], 'r.-', ms=4, lw=1)
    ax.plot(*x0, 'ko', ms=7, label='x0')
    ax.plot(*x_exact, 'g*', ms=14, label='x*')
    ax.set_title(name); ax.set_xlabel('x1'); ax.set_ylabel('x2'); ax.legend(fontsize=8)
fig.suptitle(f'Варіант {VARIANT}: траєкторії пошуку')
fig.tight_layout()
fig.savefig('trajectories.png', dpi=150)
fig

## 7. Зведена порівняльна таблиця

In [ ]:
summary = pd.DataFrame([
    dict(method="Покоординатний спуск", x1=x_cd[0], x2=x_cd[1], f=f_vec(x_cd), iters=it_cd, calls=calls_cd, time=t_cd),
    dict(method="Хука-Дживса", x1=x_hj[0], x2=x_hj[1], f=f_vec(x_hj), iters=it_hj, calls=calls_hj, time=t_hj),
    dict(method="Нелдера-Міда", x1=res_nm.x[0], x2=res_nm.x[1], f=res_nm.fun, iters=res_nm.nit, calls=calls_nm, time=t_nm),
])
summary

## 8. Чутливість до початкової точки

Запускаємо всі три методи з трьох різних x0 і перевіряємо, чи збігаються вони до того самого мінімуму.

In [ ]:
starts = [np.array(x0), np.array([0.0, 0.0]), np.array([2.0, -1.0])]
rows = []
for s in starts:
    xc, _, _ = coordinate_descent(f_vec, s, eps=eps0)
    xh, _, _ = hooke_jeeves(f_vec, s, h=h0, eps=eps0)
    rn = minimize(f_vec, s, method='Nelder-Mead', options={'xatol': eps0, 'fatol': eps0})
    rows.append(dict(x0=tuple(s), cd=tuple(np.round(xc, 4)), hj=tuple(np.round(xh, 4)), nm=tuple(np.round(rn.x, 4))))
pd.DataFrame(rows)

## 9. Чутливість методу Хука-Дживса до початкового кроку h

In [ ]:
h_rows = []
for h_test in (0.1, 0.5, 1.0):
    calls["n"] = 0
    xh, it_, _ = hooke_jeeves(f_vec, x0, h=h_test, eps=eps0)
    h_rows.append(dict(h=h_test, x1=xh[0], x2=xh[1], f=f_vec(xh), iters=it_, calls=calls["n"]))
pd.DataFrame(h_rows)